# 04 — AIFS storm tracking

Opens the AIFS forecast from the icechunk store, extracts the MSL field,
saves it as a local NetCDF, and runs PyStormTracker (same algorithm as ERA5)
to produce a tracks CSV for each initialisation date.

In [ ]:
import datetime as dt
import os
import pathlib
import tempfile

import icechunk
import pandas as pd
import pystormtracker as pst
import xarray as xr

In [ ]:
storm_name = "claudia"
init_date_str = "2025110700"  # %Y%m%d%H
n_members = 0

storage_bucket = "aifs-modal-unibe"
outputs_prefix = "aifs-outputs-det"

msl_nc_file = "../data/interim/aifs_msl_claudia_2025110700.nc"
output_file = "../data/interim/aifs_tracks_claudia_2025110700.csv"

In [ ]:
# process params
start_date = dt.datetime.strptime(str(init_date_str), "%Y%m%d%H").replace(tzinfo=dt.UTC)
outputs_branch = f"{storm_name}-{init_date_str}"
zarr_group = f"{start_date.strftime('%Y-%m-%d')}/{start_date.strftime('%H')}z"

print(f"Opening  : branch={outputs_branch}  group={zarr_group}")

outputs_storage = icechunk.tigris_storage(
    bucket=storage_bucket,
    prefix=outputs_prefix,
    region=os.getenv("AWS_REGION", None),
    access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)
outputs_repo = icechunk.Repository.open(outputs_storage)
session = outputs_repo.readonly_session(outputs_branch)

ds = xr.open_dataset(
    session.store,
    group=zarr_group,
    engine="zarr",
    zarr_format=3,
)
print(f"AIFS output: {list(ds.data_vars)}")
print(f"  dims: {dict(ds.sizes)}")

In [ ]:
all_tracks = []
msl_out_path = pathlib.Path(msl_nc_file)

# Build (member_id, msl_da) pairs — deterministic has no ensemble_member dim
if n_members == 0:
    members = [(None, ds["msl"].load())]
else:
    members = [
        (m, ds["msl"].sel(ensemble_member=m).drop_vars("ensemble_member").load())
        for m in range(n_members)
    ]

for i, (member, msl_da) in enumerate(members):
    label = "det" if member is None else member
    print(f"\n--- Member {label} ---")

    # First item: keep as Snakemake-tracked output; others: temp file
    if i == 0:
        nc_path = msl_out_path
        msl_da.to_netcdf(nc_path)
        print(f"  Saved MSL → {nc_path}  ({nc_path.stat().st_size / 1e6:.0f} MB)")
    else:
        _tmp = tempfile.NamedTemporaryFile(suffix=".nc", delete=False)
        nc_path = pathlib.Path(_tmp.name)
        _tmp.close()
        msl_da.to_netcdf(nc_path)

    tracker = pst.HodgesTracker()
    tracks = tracker.track(nc_path, varname="msl", mode="min", filter=True)
    print(f"  Detected {len(tracks)} tracks")

    if len(tracks) > 0:
        member_df = pd.DataFrame(
            {
                "track_id": tracks.track_ids,
                "time": pd.to_datetime(tracks.times),
                "latitude": tracks.lats,
                "longitude": tracks.lons,
                "msl_filtered": tracks.vars["msl_spectral_filtered"],
            }
        )
        member_df["longitude"] = ((member_df["longitude"] + 180) % 360) - 180
        member_df["member"] = member
        member_df["init_date"] = str(init_date_str)
        all_tracks.append(member_df)

    if i != 0:
        nc_path.unlink()

tracks_df = pd.concat(all_tracks, ignore_index=True) if all_tracks else pd.DataFrame()
print(f"\nTotal: {tracks_df['track_id'].nunique()} tracks, {len(tracks_df)} points")
tracks_df.head()

In [ ]:
output_path = pathlib.Path(output_file)
tracks_df.to_csv(output_path, index=False)
print(f"Saved → {output_path}")